# Sprint Notebook — Final 2-3 Day Push

Baseline: **v7 Platform 0.5636**, OOF 0.6429, M1-5 OOF 0.6515  
Sections: A · B · C · D · E · F · G · H

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import os

SEED       = 42
DATA_DIR   = '../data/'
MODEL_DIR  = '../models/'
SUBMIT_DIR = '../submissions/'

np.random.seed(SEED)
print('Imports OK')

Imports OK


In [2]:
# Load train and test data (needed for month_of_year mask and y)
print('Loading data...')
train_df = pd.read_parquet(f'{DATA_DIR}train_features_tier2.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}test_features_tier2.parquet')
y = train_df['invalid_ratio'].values
print(f'  Train: {train_df.shape}  Test: {test_df.shape}')
print(f'  Train months: {sorted(train_df["month_of_year"].unique())}')
print(f'  Test  months: {sorted(test_df["month_of_year"].unique())}')

Loading data...


  Train: (6076546, 31)  Test: (2028750, 30)
  Train months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  Test  months: [1, 2, 3, 4, 5]


---
## Section A — M1-5 Weight Optimization + Fine-Grained Search

**Rationale**: v7 weights (LGB=0.35, XGB=0.65) were optimised on full 12-month OOF
with step=0.05. The test set is M1-5 only. Re-searching on the M1-5 OOF subset
with finer step=0.01 may find weights that better match the test distribution.

Note: v8a OOF ≡ v7 OOF (step11_gpu.py trains identically to v7; only test-time TE
differs). We use v8a OOF files as the v7 OOF proxy when v7 files are absent.

In [3]:
# ── Load OOF predictions (prefer v7; fall back to v8a which is identical) ────
def load_oof(model, version_primary='v7', version_fallback='v8a'):
    primary  = f'{MODEL_DIR}{model}_oof_{version_primary}.npy'
    fallback = f'{MODEL_DIR}{model}_oof_{version_fallback}.npy'
    if os.path.exists(primary):
        arr = np.load(primary)
        print(f'  Loaded {primary}')
        return arr, version_primary
    elif os.path.exists(fallback):
        arr = np.load(fallback)
        print(f'  Loaded {fallback}  [proxy for {version_primary} — OOF is identical]')
        return arr, version_fallback
    else:
        raise FileNotFoundError(f'Neither {primary} nor {fallback} found.')

print('Loading OOF predictions...')
lgb_oof, lgb_ver = load_oof('lgb')
xgb_oof, xgb_ver = load_oof('xgb')

# CB oof v4 (optional — v7 ensemble gave it weight=0)
cb_oof_path = f'{MODEL_DIR}cb_oof_v4.npy'
cb_oof = np.load(cb_oof_path) if os.path.exists(cb_oof_path) else None
print(f'  CB v4 OOF: {"loaded" if cb_oof is not None else "not found (skip)"}')

lgb_rho = spearmanr(y, lgb_oof)[0]
xgb_rho = spearmanr(y, xgb_oof)[0]
print(f'\nFull OOF Spearman:')
print(f'  LGB ({lgb_ver}): {lgb_rho:.4f}  (expected ~0.6336)')
print(f'  XGB ({xgb_ver}): {xgb_rho:.4f}  (expected ~0.6403)')

Loading OOF predictions...
  Loaded ../models/lgb_oof_v8a.npy  [proxy for v7 — OOF is identical]
  Loaded ../models/xgb_oof_v8a.npy  [proxy for v7 — OOF is identical]
  CB v4 OOF: not found (skip)



Full OOF Spearman:
  LGB (v8a): 0.6336  (expected ~0.6336)
  XGB (v8a): 0.6403  (expected ~0.6403)


In [4]:
# ── M1-5 mask ─────────────────────────────────────────────────────────────────
m15_mask = train_df['month_of_year'].isin([1, 2, 3, 4, 5]).values
print(f'M1-5 train rows: {m15_mask.sum():,} / {len(train_df):,}')

y_m15      = y[m15_mask]
lgb_oof_m15 = lgb_oof[m15_mask]
xgb_oof_m15 = xgb_oof[m15_mask]
cb_oof_m15  = cb_oof[m15_mask] if cb_oof is not None else None

lgb_rho_m15 = spearmanr(y_m15, lgb_oof_m15)[0]
xgb_rho_m15 = spearmanr(y_m15, xgb_oof_m15)[0]
print(f'\nM1-5 OOF Spearman:')
print(f'  LGB: {lgb_rho_m15:.4f}')
print(f'  XGB: {xgb_rho_m15:.4f}')

M1-5 train rows: 2,470,429 / 6,076,546



M1-5 OOF Spearman:
  LGB: 0.6428
  XGB: 0.6482


In [5]:
# ── Fine-grained weight search (step=0.01) on M1-5 OOF ───────────────────────
from itertools import product

weights = np.arange(0, 1.01, 0.01)

best_rho_m15  = -1
best_w_m15    = None

# Search over LGB weight; XGB = 1 - lgb_w (CB=0 as in v7)
results_m15 = []
for lgb_w in weights:
    xgb_w = 1.0 - lgb_w
    pred = lgb_w * lgb_oof_m15 + xgb_w * xgb_oof_m15
    rho  = spearmanr(y_m15, pred)[0]
    results_m15.append((lgb_w, xgb_w, rho))
    if rho > best_rho_m15:
        best_rho_m15 = rho
        best_w_m15   = (lgb_w, xgb_w)

print(f'M1-5 weight search (step=0.01, CB=0) complete.')
print(f'  Best M1-5 weights : LGB={best_w_m15[0]:.2f}, XGB={best_w_m15[1]:.2f}')
print(f'  Best M1-5 OOF Rho : {best_rho_m15:.4f}')
print(f'\n  v7 weights        : LGB=0.35, XGB=0.65')
print(f'  v7 M1-5 OOF (ref) : 0.6515')

# Compare v7 weights on M1-5
pred_v7_m15 = 0.35 * lgb_oof_m15 + 0.65 * xgb_oof_m15
rho_v7_m15  = spearmanr(y_m15, pred_v7_m15)[0]
print(f'  v7 weights applied to M1-5 OOF: {rho_v7_m15:.4f}')
print(f'  Delta (new - v7): {best_rho_m15 - rho_v7_m15:+.4f}')

M1-5 weight search (step=0.01, CB=0) complete.
  Best M1-5 weights : LGB=0.39, XGB=0.61
  Best M1-5 OOF Rho : 0.6515

  v7 weights        : LGB=0.35, XGB=0.65
  v7 M1-5 OOF (ref) : 0.6515


  v7 weights applied to M1-5 OOF: 0.6515
  Delta (new - v7): +0.0000


In [6]:
# ── Also run fine-grained search on FULL OOF for comparison ──────────────────
best_rho_full = -1
best_w_full   = None
results_full  = []
for lgb_w in weights:
    xgb_w = 1.0 - lgb_w
    pred  = lgb_w * lgb_oof + xgb_w * xgb_oof
    rho   = spearmanr(y, pred)[0]
    results_full.append((lgb_w, xgb_w, rho))
    if rho > best_rho_full:
        best_rho_full = rho
        best_w_full   = (lgb_w, xgb_w)

print(f'Full OOF weight search (step=0.01, CB=0) complete.')
print(f'  Best full weights : LGB={best_w_full[0]:.2f}, XGB={best_w_full[1]:.2f}')
print(f'  Best full OOF Rho : {best_rho_full:.4f}')

print(f'\n=== Summary ===')
print(f'  Full-OOF optimal : LGB={best_w_full[0]:.2f}  XGB={best_w_full[1]:.2f}  Rho={best_rho_full:.4f}')
print(f'  M1-5-OOF optimal : LGB={best_w_m15[0]:.2f}  XGB={best_w_m15[1]:.2f}  Rho={best_rho_m15:.4f}')
print(f'  v7 weights       : LGB=0.35   XGB=0.65')

Full OOF weight search (step=0.01, CB=0) complete.
  Best full weights : LGB=0.36, XGB=0.64
  Best full OOF Rho : 0.6429

=== Summary ===
  Full-OOF optimal : LGB=0.36  XGB=0.64  Rho=0.6429
  M1-5-OOF optimal : LGB=0.39  XGB=0.61  Rho=0.6515
  v7 weights       : LGB=0.35   XGB=0.65


In [7]:
# ── Optional: 3-model search including CB (if available) ─────────────────────
if cb_oof is not None:
    best_rho_3m = -1
    best_w_3m   = None
    step = 0.05  # coarser for 3-model to keep it fast
    ws   = np.arange(0, 1.01, step)
    for lgb_w in ws:
        for xgb_w in ws:
            cb_w = round(1.0 - lgb_w - xgb_w, 4)
            if cb_w < 0 or cb_w > 1:
                continue
            pred = lgb_w * lgb_oof_m15 + xgb_w * xgb_oof_m15 + cb_w * cb_oof_m15
            rho  = spearmanr(y_m15, pred)[0]
            if rho > best_rho_3m:
                best_rho_3m = rho
                best_w_3m   = (lgb_w, xgb_w, cb_w)
    print(f'3-model M1-5 search (step=0.05):')
    print(f'  Best: LGB={best_w_3m[0]:.2f}, XGB={best_w_3m[1]:.2f}, CB={best_w_3m[2]:.2f}')
    print(f'  M1-5 Rho: {best_rho_3m:.4f}')
else:
    print('CB OOF not available — skipping 3-model search.')
    best_w_3m = None

CB OOF not available — skipping 3-model search.


In [8]:
# ── Generate ensemble_v12.csv if test predictions are available ───────────────
# Prefer v7 test preds (full-data TE); fall back instructions if missing.

lgb_test_path_v7  = f'{MODEL_DIR}lgb_test_v7.npy'
xgb_test_path_v7  = f'{MODEL_DIR}xgb_test_v7.npy'
lgb_test_path_v8a = f'{MODEL_DIR}lgb_test_v8a.npy'
xgb_test_path_v8a = f'{MODEL_DIR}xgb_test_v8a.npy'

if os.path.exists(lgb_test_path_v7) and os.path.exists(xgb_test_path_v7):
    lgb_test = np.load(lgb_test_path_v7)
    xgb_test = np.load(xgb_test_path_v7)
    test_src  = 'v7 (full-data TE)'
    use_weights = best_w_m15  # apply M1-5 optimised weights
    print(f'Using v7 test predictions ({test_src}).')
elif os.path.exists(lgb_test_path_v8a) and os.path.exists(xgb_test_path_v8a):
    lgb_test = np.load(lgb_test_path_v8a)
    xgb_test = np.load(xgb_test_path_v8a)
    test_src  = 'v8a (M1-5 TE — different from v7!)'
    use_weights = best_w_m15
    print(f'WARNING: v7 test preds not found; using v8a ({test_src}).')
    print('  v8a test TE differs from v7; the v12 submission may not outperform v7.')
    print('  To generate a clean v12, download lgb_test_v7.npy + xgb_test_v7.npy from GPU server.')
else:
    lgb_test = None
    print('ERROR: No test predictions found. Download from GPU server:')
    print('  models/lgb_test_v7.npy')
    print('  models/xgb_test_v7.npy')

if lgb_test is not None:
    lgb_w, xgb_w = use_weights
    test_pred = lgb_w * lgb_test + xgb_w * xgb_test
    test_pred = np.clip(test_pred, 0, 1)

    # Verify shape matches test_df
    assert len(test_pred) == len(test_df), f'Shape mismatch: {len(test_pred)} vs {len(test_df)}'
    assert not np.isnan(test_pred).any(), 'NaN in predictions!'

    out_path = f'{SUBMIT_DIR}ensemble_v12.csv'
    pd.DataFrame({'id': test_df.index, 'invalid_ratio': test_pred}).to_csv(out_path, index=False)
    print(f'\nSaved: {out_path}')
    print(f'  Weights: LGB={lgb_w:.2f}, XGB={xgb_w:.2f}')
    print(f'  Test source: {test_src}')
    print(f'  Rows: {len(test_pred):,}  Min: {test_pred.min():.4f}  Max: {test_pred.max():.4f}')
    print(f'  Expected rows: 2,028,750')

  v8a test TE differs from v7; the v12 submission may not outperform v7.
  To generate a clean v12, download lgb_test_v7.npy + xgb_test_v7.npy from GPU server.



Saved: ../submissions/ensemble_v12.csv
  Weights: LGB=0.39, XGB=0.61
  Test source: v8a (M1-5 TE — different from v7!)
  Rows: 2,028,750  Min: 0.0052  Max: 1.0000
  Expected rows: 2,028,750
